In [1]:
#Cell 1: Raw data
raw_products = [
    {"id": 1, "name": "  Wireless Mouse ", "price": "$25.99",    "category": "Electronics", "stock": "40"},
    {"id": 2, "name": "USB Keyboard",      "price": 45,          "category": "electronics", "stock": 15},
    {"id": 3, "name": "wireless mouse",    "price": "25.99",     "category": "Electronics", "stock": "40"},  # duplicate
    {"id": 4, "name": "Notebook A5",       "price": "Rs. 1,200", "category": "Stationery",  "stock": None},
    {"id": 5, "name": "Gel Pen",           "price": None,        "category": "Stationery",  "stock": 200},   # price missing
    {"id": 6, "name": "Desk Lamp",         "price": "N/A",       "category": "",            "stock": 8},     # bad price
    {"id": 7, "name": "Water Bottle",      "price": "  12.5 ",   "category": None,          "stock": "30"},
    {"id": 8, "name": "",                  "price": 10,          "category": "Misc",        "stock": 5},     # name missing
    {"id": 9, "name": "Notebook A5",       "price": 1200.0,      "category": "STATIONERY",  "stock": 50},    # duplicate
]

In [2]:
#Helper functions
import re

def clean_price(value):
    """Kisi bhi type ki price ko float mein badlo, warna None."""
    if isinstance(value, bool) or value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        match = re.search(r"\d[\d,]*\.?\d*", value)   # e.g. "Rs. 1,200" -> "1,200"
        if match:
            return float(match.group().replace(",", ""))
    return None

def clean_text(value, default=None):
    """Extra spaces hatao; khali ho to default do."""
    if value is None:
        return default
    value = str(value).strip()
    return value if value else default

def clean_stock(value):
    """Stock ko int banao, missing ho to 0."""
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return 0

In [3]:
#Cleaning + duplicate removal
cleaned = []
seen_names = set()
dropped = []   # (id, reason)

for item in raw_products:
    name = clean_text(item.get("name"))
    price = clean_price(item.get("price"))

    if name is None:
        dropped.append((item["id"], "missing name"))
        continue
    if price is None:
        dropped.append((item["id"], "missing/invalid price"))
        continue

    key = name.lower()
    if key in seen_names:
        dropped.append((item["id"], "duplicate"))
        continue
    seen_names.add(key)

    cleaned.append({
        "id": item["id"],
        "name": name.title(),
        "price": price,
        "category": clean_text(item.get("category"), "Uncategorized").title(),
        "stock": clean_stock(item.get("stock")),
    })

print(f"Kept: {len(cleaned)} | Dropped: {len(dropped)}")

Kept: 4 | Dropped: 5


In [4]:
#Category ke hisaab se dictionary
from collections import defaultdict

catalogue = defaultdict(list)
for product in cleaned:
    catalogue[product["category"]].append(product)

catalogue = dict(catalogue)   # normal dict

In [5]:
#Neat summary
print("=" * 55)
print("PRODUCT CATALOGUE SUMMARY".center(55))
print("=" * 55)

for category in sorted(catalogue):
    items = catalogue[category]
    prices = [p["price"] for p in items]
    print(f"\n{category}  ({len(items)} items | avg {sum(prices)/len(prices):,.2f})")
    print("-" * 55)
    for p in sorted(items, key=lambda x: x["name"]):
        print(f"  #{p['id']:<3} {p['name']:<20} {p['price']:>10,.2f}   stock: {p['stock']}")

print("\n" + "=" * 55)
print(f"Total products kept : {len(cleaned)}")
print(f"Categories          : {len(catalogue)}")
print(f"Rows dropped        : {len(dropped)}")
for pid, reason in dropped:
    print(f"   - id {pid}: {reason}")
print("=" * 55)

               PRODUCT CATALOGUE SUMMARY               

Electronics  (2 items | avg 35.49)
-------------------------------------------------------
  #2   Usb Keyboard              45.00   stock: 15
  #1   Wireless Mouse            25.99   stock: 40

Stationery  (1 items | avg 1,200.00)
-------------------------------------------------------
  #4   Notebook A5            1,200.00   stock: 0

Uncategorized  (1 items | avg 12.50)
-------------------------------------------------------
  #7   Water Bottle              12.50   stock: 30

Total products kept : 4
Categories          : 3
Rows dropped        : 5
   - id 3: duplicate
   - id 5: missing/invalid price
   - id 6: missing/invalid price
   - id 8: missing name
   - id 9: duplicate
